# CT Scanner Simulator

**Group:**
* Jakub Biernat 160248
* Eryk Masian 160228

**Tomograph Model:**
* **Cone-beam**

**Technologies Used:**
* **Language:** Python
* **Libraries:** numpy, matplotlib, ipywidgets, pydicom

**Functions**
* [Radon Transform](../src/radon_transform.py)
* [Emitter Positioning](../src/emiters.py)
* [Detector Positioning](../src/detectors.py)
* [Bresenham Algorithm]()

In [4]:
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
import io

from networkx.algorithms.sparsifiers import spanner

# ---------------- OBRAZY ----------------
available_images = {
    "CT_ScoutView": "../data/input/CT_ScoutView.jpg",
    "CT_ScoutView-large": "../data/input/CT_ScoutView-large.jpg",
    "Kolo": "../data/input/Kolo.jpg",
    "Kropka": "../data/input/Kropka.jpg",
    "Kwadraty2": "../data/input/Kwadraty2.jpg",
    "Paski2": "../data/input/Paski2.jpg",
    "SADDLE_PE": "../data/input/SADDLE_PE.jpg",
    "SADDLE_PE-large": "../data/input/SADDLE_PE-large.jpg",
    "Shepp_logan": "../data/input/Shepp_logan.jpg",
    "CT_ScoutView_DCM": "../data/input/CT_ScoutView.dcm",
    "CT_ScoutView-large_DCM": "../data/input/CT_ScoutView-large.dcm",
    "Kolo_DCM": "../data/input/Kolo.dcm",
    "Kropka_DCM": "../data/input/Kropka.dcm",
    "Kwadraty2_DCM": "../data/input/Kwadraty2.dcm",
    "Paski2_DCM": "../data/input/Paski2.dcm",
    "SADDLE_PE_DCM": "../data/input/SADDLE_PE.dcm",
    "SADDLE_PE-large_DCM": "../data/input/SADDLE_PE-large.dcm",
    "Shepp_logan_DCM": "../data/input/Shepp_logan.dcm"
}

# ---------------- STAŁE ----------------
FIGSIZE = (6, 6)
DPI = 150
PREVIEW_EMITTER_POS = 90  # kąt emitera do preview

# ---------------- UI ----------------
image_selector = widgets.Dropdown(
    options=list(available_images.keys()),
    value=list(available_images.keys())[0],
    description="Obraz:"
)

detectors_slider = widgets.IntSlider(
    value=90, min=30, max=720, step=30,
    description="Detektory:"
)

span_slider = widgets.FloatSlider(
    value=30, min=15, max=270, step=15,
    description="Rozpiętość:"
)

scans_slider = widgets.IntSlider(
    value=360,
    min=90,
    max=720,
    step=90,
    description="Skany:"
)

alpha_label = widgets.Label()
canvas = widgets.Image(format="png")
info_out = widgets.Output()

# 🔥 STAŁY ROZMIAR WIDŻETU
canvas.layout = widgets.Layout(
    width="300px",
    height="300px"
)

left_panel = widgets.VBox([
    image_selector,
    detectors_slider,
    span_slider,
    widgets.HBox([scans_slider, alpha_label])
], layout=widgets.Layout(width="30%"))

middle_panel = widgets.VBox([
    canvas
], layout=widgets.Layout(
    width="40%",
    align_items="center",
    justify_content="center"
))

right_panel = widgets.VBox([
    info_out
], layout=widgets.Layout(width="30%"))

ui = widgets.HBox([
    left_panel,
    middle_panel,
    right_panel
])

display(ui)

# ---------------- HELPERS ----------------
def fig_to_png(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=DPI)
    buf.seek(0)
    return buf.read()

# ---------------- UPDATE ----------------
def update(*args):
    name = image_selector.value
    path = available_images[name]

    info, image = load_image(path)

    detectors = detectors_slider.value
    span = span_slider.value

    xe, ye = emitter_pos(image, PREVIEW_EMITTER_POS)
    dets = detectors_pos(image, PREVIEW_EMITTER_POS, detectors, span)

    scans = scans_slider.value
    angle = 360 / scans  # α

    alpha_label.value = f"α = {angle:.3f}°"

    # ---------------- RYSOWANIE ----------------
    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)

    ax.imshow(image, cmap="gray", aspect="equal")

    # emitter
    ax.scatter(xe, ye, c="red", s=80)

    # detektory + linie
    if len(dets) > 0:
        dx, dy = zip(*dets)
        ax.scatter(dx, dy, c="blue", s=30)

        for x, y in dets:
            ax.plot([xe, x], [ye, y],
                    c="red", linewidth=0.5, alpha=0.6)

    ax.set_title(name)
    ax.axis("off")

    # ---------------- UPDATE WIDŻETA ----------------
    canvas.value = fig_to_png(fig)
    plt.close(fig)

    # ---------------- INFO ----------------
    with info_out:
        info_out.clear_output(wait=True)
        print("=== INFO DICOM ===")
        for k, v in info.items():
            print(f"{k}: {v}")

# ---------------- OBSERWATORY ----------------
for w in [image_selector, detectors_slider, span_slider, scans_slider]:
    w.observe(update, "value")

# start
update()

In [1]:
import os
import sys
import io
import datetime

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image as PILImage

import pydicom
from pydicom.dataset import FileDataset, FileMetaDataset
from pydicom.uid import UID

sys.path.append(os.path.abspath('..'))

from src.radon_transform import get_sinogram, inverse_radon
from src.bresenham import bresenham
from src.emiters import emitter_pos
from src.detectors import detectors_pos
from src.filter import apply_filter
from src.rmse import calculate_rmse
from src.handle_dicoms import load_image
from src.detectors import detectors_pos

In [5]:
PLOT_PX = 500
DPI = 100
FIG_IN = PLOT_PX / DPI

available_images = {
    "CT_ScoutView": "../data/input/CT_ScoutView.jpg",
    "CT_ScoutView-large": "../data/input/CT_ScoutView-large.jpg",
    "Kolo": "../data/input/Kolo.jpg",
    "Kropka": "../data/input/Kropka.jpg",
    "Kwadraty2": "../data/input/Kwadraty2.jpg",
    "Paski2": "../data/input/Paski2.jpg",
    "SADDLE_PE": "../data/input/SADDLE_PE.jpg",
    "SADDLE_PE-large": "../data/input/SADDLE_PE-large.jpg",
    "Shepp_logan": "../data/input/Shepp_logan.jpg",
    "CT_ScoutView_DCM": "../data/input/CT_ScoutView.dcm",
    "CT_ScoutView-large_DCM": "../data/input/CT_ScoutView-large.dcm",
    "Kolo_DCM": "../data/input/Kolo.dcm",
    "Kropka_DCM": "../data/input/Kropka.dcm",
    "Kwadraty2_DCM": "../data/input/Kwadraty2.dcm",
    "Paski2_DCM": "../data/input/Paski2.dcm",
    "SADDLE_PE_DCM": "../data/input/SADDLE_PE.dcm",
    "SADDLE_PE-large_DCM": "../data/input/SADDLE_PE-large.dcm",
    "Shepp_logan_DCM": "../data/input/Shepp_logan.dcm"
}